In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import torch
import torchvision
import torchvision.transforms as T
from PIL import Image
from sklearn.metrics import roc_auc_score
from torch.utils.data import DataLoader
from torch_uncertainty.methods import mc_dropout
from torchvision.models import VGG16_Weights, vgg16

%matplotlib inline
sns.set_theme(style="whitegrid")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Отдельное изображение и VGG16-ImageNet

In [ ]:
class VGGWrapper(torch.nn.Module):
    def __init__(self):
        super().__init__()

        weights = VGG16_Weights.DEFAULT
        self.model = vgg16(weights=weights).train()

        transforms = weights.transforms()
        self.normalize = T.Normalize(mean=transforms.mean, std=transforms.std)

    def forward(self, x):
        return self.model(self.normalize(x))


mc_model = mc_dropout(VGGWrapper(), num_estimators=20)
mc_model.eval();

In [ ]:
imagenet_transforms = VGG16_Weights.DEFAULT.transforms()
vis_transform = T.Compose(
    [
        T.Resize(imagenet_transforms.resize_size),
        T.CenterCrop(imagenet_transforms.crop_size),
        T.ToTensor(),
    ]
)

categories = VGG16_Weights.DEFAULT.meta["categories"]

In [ ]:
image = vis_transform(
    Image.open(Path.cwd().parent / "images" / "cat_on_chair.jpg").convert("RGB")
).unsqueeze(0)

with torch.no_grad():
    logits = mc_model(image)

probs = torch.softmax(logits, dim=-1)
p_mean = probs.mean(dim=0)

total_uncertainty = -torch.sum(p_mean * torch.log(p_mean + 1e-12), dim=-1)
aleatoric_uncertainty = -torch.sum(
    probs * torch.log(probs + 1e-12), dim=-1
).mean(dim=0)
epistemic_uncertainty = total_uncertainty - aleatoric_uncertainty

top1_prob, top1_class_idx = torch.max(p_mean, dim=0)

plt.imshow(image.squeeze(0).permute(1, 2, 0).numpy())
plt.title(
    f"Класс {categories[top1_class_idx.item()]} "
    f"с вероятностью {top1_prob.item():.2f}"
)
plt.figtext(
    0.17,
    -0.13,
    f"Максимальная неопределенность: {np.log(1000):.2f}\n"
    f"Полная неопределенность: {total_uncertainty:.2f}\n"
    f"Алеаторная неопределенность: {aleatoric_uncertainty:.2f}\n"
    f"Эпистемическая неопределенность: {epistemic_uncertainty:.2f}",
    ha="left",
    fontsize=10,
)
plt.axis("off")
plt.tight_layout()
plt.show()

На изображении с кошкой на диване алеаторная неопределенность преобладает над эпистемической, как и предполагалось изначально.

# CIFAR-10 vs. SVHN

## Загрузка датасетов

In [ ]:
cifar_transforms = T.Compose(
    [
        T.ToTensor(),
        T.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
    ]
)

cifar_test = torchvision.datasets.CIFAR10(
    root="../data", train=False, download=True, transform=cifar_transforms
)
cifar_loader = DataLoader(
    cifar_test, batch_size=128, shuffle=False, num_workers=2
)

svhn_test = torchvision.datasets.SVHN(
    root="../data", split="test", download=True, transform=cifar_transforms
)
svhn_loader = DataLoader(
    svhn_test, batch_size=128, shuffle=False, num_workers=2
)

## Оборачиваем претренированную модель

In [ ]:
base_model = torch.hub.load(
    "chenyaofo/pytorch-cifar-models", "cifar10_vgg16_bn", pretrained=True
)
mc_model = mc_dropout(base_model, num_estimators=20, last_layer=False).to(
    device
)
mc_model.eval();

## Получение неопределенностей и визуализация

In [ ]:
@torch.no_grad()
def get_uncertainties(model, dataloader, num_passes):
    total, aleatoric, epistemic = [], [], []

    for images, _ in dataloader:
        images = images.to(device)
        output = model(images).view(num_passes, images.size(0), -1)

        probs = torch.softmax(output, dim=-1)
        p_mean = probs.mean(dim=0)

        total_uncertainty = -torch.sum(
            p_mean * torch.log(p_mean + 1e-12), dim=-1
        )
        aleatoric_uncertainty = -torch.sum(
            probs * torch.log(probs + 1e-12), dim=-1
        ).mean(dim=0)
        epistemic_uncertainty = total_uncertainty - aleatoric_uncertainty

        total.append(total_uncertainty.cpu().numpy())
        aleatoric.append(aleatoric_uncertainty.cpu().numpy())
        epistemic.append(epistemic_uncertainty.cpu().numpy())

    return (
        np.concatenate(total),
        np.concatenate(aleatoric),
        np.concatenate(epistemic),
    )


total_id, aleatoric_id, epistemic_id = get_uncertainties(
    mc_model, cifar_loader, 20
)
total_ood, aleatoric_ood, epistemic_ood = get_uncertainties(
    mc_model, svhn_loader, 20
)

In [ ]:
y_true = np.concatenate(
    [np.ones_like(epistemic_ood), np.zeros_like(epistemic_id)]
)

auroc_total = roc_auc_score(y_true, np.concatenate([total_ood, total_id]))

plt.figure(figsize=(6, 5))

sns.kdeplot(
    total_id, fill=True, color="#3498db", label="CIFAR-10 (ID)", clip=(0, None)
)
sns.kdeplot(
    total_ood, fill=True, color="#e74c3c", label="SVHN (OOD)", clip=(0, None)
)

plt.title(f"Полная неопределенность\nAUROC = {auroc_total:.3f}", fontsize=14)
plt.xlabel("Значение энтропии", fontsize=12)
plt.ylabel("Плотность", fontsize=12)
plt.legend()

plt.tight_layout()
plt.show()

Видим, что, несмотря на высокий AUROC, пик для обоих типов неопределенности находится в точке 0, то есть для некоторых OOD изображений модель уверенно выдает предсказание, и из-за этого MC Dropout критикуется ([пример](https://auai.org/uai2018/proceedings/papers/207.pdf)).

In [ ]:
plt.figure(figsize=(6, 5))

plt.scatter(
    aleatoric_id,
    epistemic_id,
    alpha=0.3,
    color="#3498db",
    label="CIFAR-10 (ID)",
    s=15,
)
plt.scatter(
    aleatoric_ood,
    epistemic_ood,
    alpha=0.3,
    color="#e74c3c",
    label="SVHN (OOD)",
    s=15,
)

max_val = max(np.max(aleatoric_ood), np.max(epistemic_ood))
plt.plot(
    [0, max_val],
    [0, max_val],
    "k--",
    alpha=0.5,
    label="Эпистемическая = Алеаторной",
)

plt.title("Соотношение типов неопределенности", fontsize=14)
plt.xlabel("Алеаторная", fontsize=13)
plt.ylabel("Эпистемическая", fontsize=13)
plt.legend(fontsize=12)
plt.grid(alpha=0.3)

plt.tight_layout()
plt.show()

MC Dropout в отличие от SWAG не удается качественно отделить алеаторную неопределенность от эпистемической: первый тип преобладает как для ID изображений, так и для OOD изображений.